# 🏏 Cricbuzz LiveStats — API Data Fetching Practice

This notebook walks you through:
1. Calling the Cricbuzz RapidAPI
2. Parsing the JSON responses
3. Loading data into SQLite
4. Running SQL queries with pandas

> ⚠️ Make sure you have a valid `X_RAPIDAPI_KEY` in `utils/.env` before running API cells.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import requests
import json
import sqlite3
import pandas as pd
from dotenv import load_dotenv
import os

# Load API key from .env
load_dotenv('../utils/.env')
API_KEY = os.getenv('X_RAPIDAPI_KEY')

HEADERS = {
    'x-rapidapi-key':  API_KEY,
    'x-rapidapi-host': 'cricbuzz-cricket.p.rapidapi.com'
}
BASE_URL = 'https://cricbuzz-cricket.p.rapidapi.com'

print('API Key loaded:', bool(API_KEY))

## 1. Fetch Live Matches

In [ ]:
response = requests.get(f'{BASE_URL}/matches/v1/live', headers=HEADERS)
data = response.json()

matches = []
for type_match in data.get('typeMatches', []):
    for series in type_match.get('seriesMatches', []):
        wrapper = series.get('seriesAdWrapper', {})
        for m in wrapper.get('matches', []):
            info = m.get('matchInfo', {})
            matches.append({
                'matchId':   info.get('matchId'),
                'matchDesc': info.get('matchDesc'),
                'team1':     info.get('team1', {}).get('teamName'),
                'team2':     info.get('team2', {}).get('teamName'),
                'format':    info.get('matchFormat'),
                'status':    info.get('status'),
                'venue':     info.get('venueInfo', {}).get('ground'),
            })

df_live = pd.DataFrame(matches)
print(f'{len(df_live)} live matches found')
df_live.head()

## 2. Search for a Player

In [ ]:
player_name = 'Rohit Sharma'
response = requests.get(
    f'{BASE_URL}/stats/v1/player/search',
    headers=HEADERS,
    params={'plrN': player_name}
)
players = response.json().get('player', [])
print(f'Found {len(players)} players:')
for p in players[:5]:
    print(f"  {p.get('id')} — {p.get('name')} ({p.get('teamName')})")

## 3. Get Player Batting Stats

In [ ]:
player_id = players[0]['id'] if players else 576  # Rohit Sharma fallback
batting = requests.get(f'{BASE_URL}/stats/v1/player/{player_id}/batting', headers=HEADERS).json()

headers_row = batting.get('headers', [])
rows = batting.get('values', [])

formats = headers_row[1:]
records = []
for row in rows:
    values = row.get('values', [])
    stat = values[0] if values else ''
    for i, fmt in enumerate(formats):
        if len(values) > i + 1:
            records.append({'Stat': stat, 'Format': fmt, 'Value': values[i+1]})

df_batting = pd.DataFrame(records)
df_batting.pivot(index='Stat', columns='Format', values='Value')

## 4. Connect to SQLite and Run Queries

In [ ]:
DB_PATH = '../utils/cricBuzz.db'
conn = sqlite3.connect(DB_PATH)

# List all tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('Tables in database:')
print(tables['name'].tolist())

In [ ]:
# Example: Top 10 ODI highest scores
df = pd.read_sql_query("""
    SELECT p.playerName, hs.highestScore, bc.battingAvg, bc.hundreds
    FROM highestScoreRecords hs
    JOIN players p ON hs.playerId = p.playerId
    LEFT JOIN battingCareers bc ON hs.playerId = bc.playerId AND bc.matchFormat = 'ODI'
    WHERE hs.matchFormat = 'ODI'
    ORDER BY hs.highestScore DESC
    LIMIT 10
""", conn)
df

In [ ]:
# Example: Team wins leaderboard
df_wins = pd.read_sql_query("""
    SELECT t.teamName, COUNT(*) AS wins
    FROM matches m
    JOIN teams t ON m.winningTeamId = t.teamId
    WHERE m.matchState = 'complete'
    GROUP BY t.teamName
    ORDER BY wins DESC
    LIMIT 10
""", conn)
df_wins

In [ ]:
conn.close()
print('Done!')